### Phase 5: Uncertainty Quantification and Refutation

In this phase, the stability and reliability of the counterfactual effects are evaluated using bootstrap sampling and refutation methods.

The goal is to measure how much the estimated effects vary across different samples and to validate the robustness of the causal model.

In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import RandomForestRegressor

C:\Users\vinee\.conda\envs\ds\Lib\site-packages\sklearn\utils\_param_validation.py:14: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 2.5.3)
  from scipy.sparse import csr_matrix, issparse


In [2]:
student_df = pd.read_csv("../data/raw/student-mat.csv", sep = ";")
print("Data loaded:", student_df.shape)

Data loaded: (395, 33)


In [3]:
with open("../artifacts/phase1_data_understanding/data_schema.pkl", "rb") as f:
    SCHEMA = pickle.load(f)
with open("../artifacts/phase3_causal_estimation/causal_model.pkl", "rb") as f:
    model = pickle.load(f)
with open("../artifacts/phase3_causal_estimation/causal_effect.pkl", "rb") as f:
    estimate = pickle.load(f)
print("All artifacts loaded successfully")

All artifacts loaded successfully


C:\Users\vinee\.conda\envs\ds\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Loading Data and Models

The dataset and required components from previous phases are loaded.

These include the trained counterfactual model and dataset required for computing uncertainty.

### Bootstrap Sampling

Bootstrap sampling is used to estimate the variability of the counterfactual effect.

In each iteration, a new dataset is created by sampling with replacement. The model is then trained on this sample and the effect is computed.

This process is repeated multiple times to capture variation in the estimated effect.

In [4]:
effects = []
for i in range(100):
    sample_df = student_df.sample(frac = 1, replace = True)
    X = sample_df[SCHEMA["features"]]
    y = sample_df[SCHEMA["outcome"]]
    reg = RandomForestRegressor(random_state = i)
    reg.fit(X, y)
    sample = student_df.iloc[0:1].copy()
    cf_sample = sample.copy()
    cf_sample[SCHEMA["treatment"]] += 1
    original = reg.predict(sample[SCHEMA["features"]])[0]
    cf = reg.predict(cf_sample[SCHEMA["features"]])[0]
    effects.append(cf - original)
print("Bootstrap done")

Bootstrap done


### Bootstrap Results

The effects computed from multiple bootstrap samples are stored.

These values represent how the estimated effect changes across different resampled datasets.

In [5]:
mean_effect = np.mean(effects)
std_effect = np.std(effects)
lower = np.percentile(effects, 2.5)
upper = np.percentile(effects, 97.5)
print("Mean:", mean_effect)
print("Std:", std_effect)
print("95% CI:", (lower, upper))

Mean: 3.2824577108267543
Std: 1.2764944390539517
95% CI: (np.float64(1.1894750000000012), np.float64(5.603998593073589))


### Statistical Summary

The mean and standard deviation of the bootstrap effects are calculated.

- Mean represents the average estimated effect
- Standard deviation represents the variability of the effect

These values provide insight into the stability of the counterfactual model.

### Confidence Interval

An approximate confidence interval is computed using the bootstrap mean and standard deviation.

This interval represents the range within which the effect is expected to lie under different sampling conditions.

### Refutation Tests

Refutation tests are performed to validate the robustness of the causal model.

These tests introduce controlled perturbations to check whether the estimated effect remains stable.

In [6]:
identified_estimand = model.identify_effect()
refute1 = model.refute_estimate(identified_estimand, estimate, method_name = "random_common_cause")
print("Random Common Cause Refuter:")
print(refute1)
refute2 = model.refute_estimate(identified_estimand, estimate, method_name = "data_subset_refuter")
print("Data Subset Refuter:")
print(refute2)

Random Common Cause Refuter:
Refute: Add a random common cause
Estimated effect:0.2033604613575264
New effect:0.2033604613575247
p value:0.0

Data Subset Refuter:
Refute: Use a subset of data
Estimated effect:0.2033604613575264
New effect:0.19541917171985695
p value:0.88



### Random Common Cause Test

A random variable is introduced into the model to simulate the presence of an unobserved confounder.

The stability of the estimated effect under this condition indicates robustness against hidden biases.

### Data Subset Test

The model is evaluated on a subset of the dataset.

If the estimated effect remains consistent across subsets, it indicates that the model is stable and not dependent on specific samples.

In [7]:
with open("../artifacts/phase5_uncertainty_quantification/bootstrap_effects.pkl", "wb") as f:
    pickle.dump(effects, f)
with open("../artifacts/phase5_uncertainty_quantification/bootstrap_summary.pkl", "wb") as f:
    pickle.dump({"mean" : mean_effect, "std" : std_effect, "ci" : (lower, upper)}, f)
with open("../artifacts/phase5_uncertainty_quantification/refutation_results.pkl", "wb") as f:
    pickle.dump({"random_common_cause" : refute1, "data_subset" : refute2}, f)

### Saving Results

The bootstrap effects, summary statistics, and refutation results are saved for use in later phases.

This ensures reproducibility and consistency across the pipeline.

### Conclusion

This phase evaluates the reliability of the counterfactual model using bootstrap sampling and refutation methods.

The results provide insight into the variability and robustness of the estimated effects, forming a basis for evaluation in later phases.